# YV8SKFRAME - Guided Walkthrough

This notebook explains the main execution flow of the **YV8SKFRAME** project: training, validation/tuning and final inference for detecting **Picture-in-Picture (PiP) advertising keyframes** in soccer match videos.

The notebook is intended to be a readable, reproducible companion to the GitHub repository. It does not force long training runs by default; heavy cells are protected by execution flags.


## 0. Path disclaimer

The examples assume that the project folder was downloaded or cloned to the root of the `C:` drive:

```text
C:\YV8SKFRAME
```

If your repository is located elsewhere, change `ROOT_DIR` in the next cell.


In [ ]:
from pathlib import Path
import os
import sys

ROOT_DIR = Path(r"C:/YV8SKFRAME")

FR_DATASET_DIR = ROOT_DIR / "Fr_DataSet_S_K_Frame"
VI_DATASET_DIR = ROOT_DIR / "Vi_DataSet_S_K_Frame"
CODES_DIR = ROOT_DIR / "Codes"
META_DIR = ROOT_DIR / "DataSet_SoccerKeyFrame - Org"

# The repository keeps the metadata inside this folder. To avoid path errors
# when the spreadsheet has a slightly different filename, the notebook searches
# automatically for the first .xlsx file available in the folder.
xlsx_files = sorted(META_DIR.glob("*.xlsx")) if META_DIR.exists() else []
META_PATH = xlsx_files[0] if xlsx_files else META_DIR / "DataSet_SoccerKeyFrame.xlsx"

TRAIN_DIR = FR_DATASET_DIR / "train"
VALID_VIDEO_DIR = VI_DATASET_DIR / "VALID"
TEST_VIDEO_DIR = VI_DATASET_DIR / "TEST"
DATA_YAML = FR_DATASET_DIR / "data.yaml"

print("ROOT_DIR:", ROOT_DIR)
print("DATA_YAML:", DATA_YAML)
print("META_DIR:", META_DIR)
print("META_PATH:", META_PATH)


## 1. Environment setup

Install the required packages before running the pipeline. If you are using Anaconda, activate your environment first. For GPU training, install a CUDA-compatible PyTorch build according to your local CUDA version.


In [ ]:
# Run this cell only if the packages are not installed.
# !pip install ultralytics opencv-python pandas numpy matplotlib openpyxl torch torchvision transformers pillow

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

try:
    import torch
    from ultralytics import YOLO
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Import error:", e)

print("NumPy:", np.__version__)

## 2. Check the expected folders

This cell verifies whether the core folders exist. It is useful before training or running inference because most errors in this project are caused by path mismatches.


In [ ]:
paths_to_check = {
    "Frame dataset": FR_DATASET_DIR,
    "Training frames": TRAIN_DIR,
    "Validation videos": VALID_VIDEO_DIR,
    "Test videos": TEST_VIDEO_DIR,
    "Codes": CODES_DIR,
    "data.yaml": DATA_YAML,
    "Ground-truth folder": META_DIR,
    "Ground-truth spreadsheet": META_PATH,
}

for name, path in paths_to_check.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8s} | {name:24s} | {path}")

print("
Note: VALID and TEST may appear as MISSING until the external video folders are downloaded from Google Drive.")
print("The metadata spreadsheet is searched automatically inside the ground-truth folder.")


## 3. Training stage

Use:

```text
C:\YV8SKFRAME\Fr_DataSet_S_K_Frame\train
C:\YV8SKFRAME\Codes\train
```

The YOLO models are trained for 50 epochs using the same image size and batch size to keep the comparison consistent.

The training cells below are disabled by default. Change `RUN_TRAINING` to `True` only when you really want to start training.


In [ ]:
RUN_TRAINING = False

YOLO_EPOCHS = 50
YOLO_IMGSZ = (640, 360)
YOLO_BATCH = 16
TRAIN_PROJECT = FR_DATASET_DIR / "training_results"

print("Training output directory:", TRAIN_PROJECT)

In [ ]:
# YOLOv8 training
if RUN_TRAINING:
    model = YOLO("yolov8n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        project=str(TRAIN_PROJECT),
        name="yolov8n_Soccer-Key-Frames"
    )
else:
    print("Training skipped. Set RUN_TRAINING=True to train YOLOv8.")

In [ ]:
# YOLO26 training
if RUN_TRAINING:
    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        project=str(TRAIN_PROJECT),
        name="yolo26n_Soccer-Key-Frames"
    )
else:
    print("Training skipped. Set RUN_TRAINING=True to train YOLO26.")

## 4. Validation and temporal post-processing

Use:

```text
C:\YV8SKFRAME\Vi_DataSet_S_K_Frame\VALID
C:\YV8SKFRAME\Codes\valid
```

The visual models produce frame-level outputs. Temporal post-processing converts those outputs into PiP segments by applying smoothing, entry/exit thresholds and minimum duration/gap rules.


In [ ]:
TEMPORAL_PARAMETERS = pd.DataFrame([
    {"parameter": "threshold_entry", "YOLOv8": 0.90, "YOLO26": 0.30, "ViT": 0.40},
    {"parameter": "threshold_exit", "YOLOv8": 0.98, "YOLO26": 0.98, "ViT": 0.94},
    {"parameter": "min_frames_entry", "YOLOv8": 15, "YOLO26": 2, "ViT": 15},
    {"parameter": "min_frames_exit", "YOLOv8": 3, "YOLO26": 3, "ViT": 3},
    {"parameter": "smoothing_window", "YOLOv8": 5, "YOLO26": 5, "ViT": 5},
    {"parameter": "min_duration_sec", "YOLOv8": 2.0, "YOLO26": 2.0, "ViT": 2.0},
    {"parameter": "min_gap_sec", "YOLOv8": 2.0, "YOLO26": 2.0, "ViT": 2.0},
])

TEMPORAL_PARAMETERS

In [ ]:
from collections import deque

def frame_to_dropframe_tc(frame_number, fps=29.97):
    """Convert a frame index to drop-frame timecode HH:MM:SS;FF."""
    nominal_fps = 30
    drop_frames = 2
    frames_per_hour = 107892
    frames_per_24_hours = frames_per_hour * 24
    frames_per_10_minutes = 17982
    frames_per_minute = 1798

    frame_number = int(round(frame_number)) % frames_per_24_hours
    d = frame_number // frames_per_10_minutes
    m = frame_number % frames_per_10_minutes
    total_minutes = d * 10 + m // frames_per_minute
    dropped = drop_frames * (total_minutes - total_minutes // 10)
    frame_number_adjusted = frame_number + dropped

    hours = frame_number_adjusted // (nominal_fps * 60 * 60)
    minutes = (frame_number_adjusted // (nominal_fps * 60)) % 60
    seconds = (frame_number_adjusted // nominal_fps) % 60
    frames = frame_number_adjusted % nominal_fps
    return f"{hours:02d}:{minutes:02d}:{seconds:02d};{frames:02d}"


def get_detected_segments(detections, fps, min_duration_sec=2.0, min_gap_sec=2.0, print_segments=True):
    """Convert frame-level detections into temporal PiP segments."""
    segments = []
    in_segment = False
    start_idx = None

    for i, (idx, detected, *_) in enumerate(detections):
        if detected and not in_segment:
            in_segment = True
            start_idx = idx
        elif not detected and in_segment:
            end_idx = detections[i - 1][0]
            segments.append((start_idx, end_idx))
            in_segment = False

    if in_segment:
        segments.append((start_idx, detections[-1][0]))

    if not segments:
        if print_segments:
            print("Without PiP")
        return []

    # Merge close segments
    merged = [segments[0]]
    for start, end in segments[1:]:
        prev_start, prev_end = merged[-1]
        gap_sec = (start - prev_end) / fps
        if gap_sec < min_gap_sec:
            merged[-1] = (prev_start, end)
        else:
            merged.append((start, end))

    # Remove short segments
    final_segments = []
    for start, end in merged:
        if (end - start) / fps >= min_duration_sec:
            final_segments.append((start, end))

    if print_segments:
        if not final_segments:
            print("Without PiP")
        for start, end in final_segments:
            print(f"PiP from {frame_to_dropframe_tc(start, fps)} to {frame_to_dropframe_tc(end, fps)}")

    return final_segments

## 5. Final inference on the test subset

Use:

```text
C:\YV8SKFRAME\Vi_DataSet_S_K_Frame\TEST
C:\YV8SKFRAME\Codes\detect
```

The cell below shows the YOLO-style inference function. It reads every frame, obtains the maximum confidence for class `0`, smooths it over time, and applies a state machine to decide when the PiP segment starts and ends.


In [ ]:
def process_video_for_PiP(
    video_path,
    model,
    threshold_entry=0.90,
    threshold_exit=0.98,
    min_frames_entry=15,
    min_frames_exit=3,
    smoothing_window=5,
):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_detections = []
    frame_idx = 0
    state = "NO_PIP"
    entry_counter = 0
    exit_counter = 0
    conf_buffer = deque(maxlen=smoothing_window)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_resized = cv2.resize(frame, (640, 360))
        results = model(frame_resized, verbose=False)

        max_conf = 0.0
        for result in results:
            boxes = result.boxes
            if boxes is None:
                continue
            for box in boxes:
                if int(box.cls.item()) == 0:
                    max_conf = max(max_conf, float(box.conf.item()))

        conf_buffer.append(max_conf)
        smooth_conf = sum(conf_buffer) / len(conf_buffer)

        detected = False
        if state == "NO_PIP":
            if smooth_conf >= threshold_entry:
                entry_counter += 1
                if entry_counter >= min_frames_entry:
                    state = "IN_PIP"
                    detected = True
                    exit_counter = 0
            else:
                entry_counter = 0
        else:
            detected = True
            if smooth_conf < threshold_exit:
                exit_counter += 1
                if exit_counter >= min_frames_exit:
                    state = "NO_PIP"
                    detected = False
                    entry_counter = 0
            else:
                exit_counter = 0

        frame_detections.append((frame_idx, detected, frame_to_dropframe_tc(frame_idx, fps)))
        frame_idx += 1

    cap.release()
    return frame_detections, fps

In [ ]:
RUN_INFERENCE = False

MODEL_NAME = "YOLO26"  # Choose: "YOLOv8" or "YOLO26"

MODEL_CONFIG = {
    "YOLOv8": {
        "weights": FR_DATASET_DIR / "training_results" / "yolov8n_Soccer-Key-Frames" / "weights" / "best.pt",
        "threshold_entry": 0.90,
        "threshold_exit": 0.98,
        "min_frames_entry": 15,
        "min_frames_exit": 3,
        "out_dir": FR_DATASET_DIR / "YOLOv8_Detect_results",
    },
    "YOLO26": {
        "weights": FR_DATASET_DIR / "training_results" / "yolo26n_Soccer-Key-Frames" / "weights" / "best.pt",
        "threshold_entry": 0.30,
        "threshold_exit": 0.98,
        "min_frames_entry": 2,
        "min_frames_exit": 3,
        "out_dir": FR_DATASET_DIR / "YOLO26_Detect_results",
    },
}

cfg = MODEL_CONFIG[MODEL_NAME]
print("Selected model:", MODEL_NAME)
print("Weights:", cfg["weights"])
print("Output directory:", cfg["out_dir"])

In [ ]:
if RUN_INFERENCE:
    model = YOLO(str(cfg["weights"]))
    results = {}

    for video_path in sorted(TEST_VIDEO_DIR.glob("*")):
        if video_path.suffix.lower() not in [".mp4", ".avi", ".mov", ".mkv"]:
            continue

        print(f"\nProcessing: {video_path.name}")
        detections, fps = process_video_for_PiP(
            video_path=video_path,
            model=model,
            threshold_entry=cfg["threshold_entry"],
            threshold_exit=cfg["threshold_exit"],
            min_frames_entry=cfg["min_frames_entry"],
            min_frames_exit=cfg["min_frames_exit"],
            smoothing_window=5,
        )

        segments = get_detected_segments(
            detections=detections,
            fps=fps,
            min_duration_sec=2.0,
            min_gap_sec=2.0,
            print_segments=True,
        )

        results[video_path.name] = {
            "fps": fps,
            "segments_frames": segments,
            "segments_tc": [(frame_to_dropframe_tc(s, fps), frame_to_dropframe_tc(e, fps)) for s, e in segments],
        }
else:
    print("Inference skipped. Set RUN_INFERENCE=True to run detection on test videos.")

## 6. Exporting reports

The detection scripts generate Excel files with detected vs. ground-truth start/end keyframes and FP/FN deviations. This allows direct comparison between models without manually inspecting each video.


In [ ]:
# This is a simplified template for exporting detections.
# The full scripts in Codes/detect include the complete comparison logic.

if False:  # change to True only after running inference and creating `results`
    cfg["out_dir"].mkdir(parents=True, exist_ok=True)
    rows = []
    for video_file, info in results.items():
        for start_tc, end_tc in info["segments_tc"]:
            rows.append({
                "video_file": video_file,
                "start_detect": start_tc,
                "end_detect": end_tc,
                "fps": info["fps"],
            })

    df = pd.DataFrame(rows)
    out_path = cfg["out_dir"] / f"detected_segments_{MODEL_NAME.lower()}.xlsx"
    df.to_excel(out_path, index=False)
    print("Saved:", out_path)

## 7. Final results summary

In the reported experiments, the three models produced no false positives. The main difference appeared in temporal recall and F1-score.


In [ ]:
summary = pd.DataFrame([
    {"Model": "YOLOv8", "Precision (%)": 100.0, "Recall (%)": 50.0, "F1 (%)": 66.7, "mAP@0.5 (%)": 100.0},
    {"Model": "YOLO26", "Precision (%)": 100.0, "Recall (%)": 51.7, "F1 (%)": 68.1, "mAP@0.5 (%)": 100.0},
    {"Model": "ViT", "Precision (%)": 100.0, "Recall (%)": 52.0, "F1 (%)": 68.4, "mAP@0.5 (%)": 100.0},
])
summary

## 8. Suggested usage

1. First, inspect the folder paths.
2. Train only when the dataset and `data.yaml` are correct.
3. Tune temporal parameters on the validation video subset.
4. Freeze the parameters.
5. Run final inference on the test subset.
6. Export and compare the spreadsheets.

This order avoids test-set leakage and keeps the model comparison fair.
